In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,when,to_timestamp,current_timestamp,lit,upper,coalesce,date_format
from pyspark.sql.types import DecimalType ,TimestampType ,StringType

StatementMeta(sparkpoolbank, 84, 2, Finished, Available, Finished, False)

In [3]:
spark.conf.set("spark.sql.parquet.outputTimestampType", "INT96")
spark.conf.set("spark.sql.parquet.writelegacyformat", "true")

bronze_path = "abfss://bronze@stbankingretail.dfs.core.windows.net/Transaction_Logs/Transaction_Logs.json"

df_bronze = spark.read.option("multiline", "true").json(bronze_path)

# 1. Data Cleansing and Standardization Transformations
df_silver = (
    df_bronze.filter(
        col("Transaction_ID").isNotNull()
        & (col("Account_Number") != "ACC_0000")
    )
    .withColumn("Txn_Amount", col("Txn_Amount").cast(DecimalType(18, 2)))
    .withColumn("Txn_Amount", coalesce(col("Txn_Amount"), lit(0.00)))
    .withColumn(
        "Txn_Type_Clean",
        when(upper(col("Txn_Type")).isin("CR", "CREDIT"), "CREDIT")
        .when(upper(col("Txn_Type")).isin("DR", "DEBIT"), "DEBIT")
        .otherwise("UNKNOWN"),
    )
    .withColumn("Location_Clean", coalesce(col("Location"), lit("UNKNOWN")))
    # Explicitly cast ingestion timestamp to StringType
    .withColumn(
        "ingestion_timestamp",
        date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss").cast(
            StringType()
        ),
    )
    # Explicitly cast transaction timestamp to StringType
    .withColumn(
        "Txn_Timestamp",
        date_format(
            to_timestamp(col("Txn_Date"), "yyyy-MM-dd HH:mm:ss"),
            "yyyy-MM-dd HH:mm:ss",
        ).cast(StringType()),
    )
    .select(
        col("Transaction_ID"),
        col("Account_Number"),
        col("Txn_Amount"),
        col("Txn_Type_Clean").alias("Txn_Type"),
        col("Txn_Timestamp"),
        col("Location_Clean").alias("Location"),
        col("ingestion_timestamp"),
    )
)

# 2. Silver Output Location
silver_output_path = (
    "abfss://silver@stbankingretail.dfs.core.windows.net/Transaction_Logs"
)

# 3. Save as parquet Table
df_silver.coalesce(1).write.format("parquet").option(
    "parquet.writer.version", "v1"
).mode("overwrite").option("overwriteSchema", "true").save(silver_output_path)

StatementMeta(sparkpoolbank, 84, 3, Finished, Available, Finished, False)

In [ ]:
#df_silver.printSchema()
# df_silver.select(
 #   "Transaction_ID",
 #   "Txn_Timestamp",
 #   "ingestion_timestamp"
# ).show(truncate=False)